# Lab 03-01: LangChain on Databricks

**Module:** 03 -- Application Development (30% of exam)  
**UI Sections:** Playground | Catalog | Serving  
**Estimated Time:** 45--60 minutes  
**Difficulty:** Intermediate

---

## Learning Objectives

By the end of this lab you will be able to:

- Use `ChatDatabricks` as the native LangChain LLM wrapper for Databricks endpoints
- Build chains using LCEL (LangChain Expression Language) pipe syntax: `prompt | llm | parser`
- Create reusable prompt templates with `ChatPromptTemplate`
- Parse LLM output with `StrOutputParser` and `JsonOutputParser`
- Construct a RAG-style chain with `RunnablePassthrough`
- Stream token-by-token output for real-time UX

---

## Exam Objectives Covered

- Identify LangChain as the primary framework for building GenAI apps on Databricks
- Use `ChatDatabricks` to connect LangChain to Databricks model serving endpoints
- Build chains using LCEL pipe syntax (`|`)
- Distinguish between output parser types
- Understand how retrievers integrate into LangChain chains

## What Is LangChain and Why Does It Matter on Databricks?

**LangChain is the primary framework tested on the exam for building GenAI applications on Databricks.** It provides a standardized way to compose LLM-powered pipelines from reusable components -- prompts, models, parsers, retrievers, and tools -- without writing custom glue code for each step.

At its core, LangChain provides:

| Concept | What It Does | Why It Matters |
|---|---|---|
| **LCEL (LangChain Expression Language)** | A declarative syntax for composing chains using the pipe operator `\|` | You describe *what* the chain does, not *how* to wire it together. The exam tests this heavily. |
| **Prompt Templates** | Reusable prompt structures with variables | Separate prompt logic from application logic. Change prompts without changing code. |
| **Output Parsers** | Convert raw LLM text into structured Python objects | Production pipelines need strings, dicts, or Pydantic objects -- not raw AI messages. |
| **Retrievers** | Fetch relevant documents from a vector store or search index | The bridge between LangChain and RAG. Module 04 covers this in depth. |

### Why ChatDatabricks?

Databricks provides `ChatDatabricks` as a **native LangChain integration** in the `databricks-langchain` package. It wraps any Databricks model serving endpoint (Foundation Model APIs, custom models, external models) into a LangChain-compatible `ChatModel`.

Key advantages of `ChatDatabricks`:
- **Automatic authentication** -- uses notebook credentials (`DATABRICKS_HOST` and `DATABRICKS_TOKEN`), no API keys to manage
- **Works with any serving endpoint** -- Foundation Model APIs, fine-tuned models, external models behind AI Gateway
- **Drop-in LangChain compatibility** -- works with all LangChain components (prompts, parsers, chains, agents)
- **Built-in support for streaming**, batching, and async calls

### LCEL Pipe Syntax -- The Core Pattern

The pipe operator `|` is how you compose LangChain chains:

```python
chain = prompt | llm | parser
result = chain.invoke({"topic": "Unity Catalog"})
```

This reads left-to-right: the prompt formats the input, passes it to the LLM, and the parser converts the output. Each component's output becomes the next component's input. **This is the single most important pattern the exam tests in Module 03.**

## Mental Model

> **Analogy:** LangChain is like a LEGO instruction booklet for AI apps. Each brick (component) snaps together with the pipe operator `|`. The prompt template is the baseplate that shapes the structure. The output parser is the finishing piece that makes it presentable. And `ChatDatabricks` is the motor brick -- it connects your LEGO model to the Databricks engine room where the real compute happens.

**How this maps to code:**

| LEGO Concept | LangChain Equivalent | What It Does |
|---|---|---|
| Instruction booklet | LCEL chain definition | Describes the assembly sequence |
| Baseplate | `ChatPromptTemplate` | Shapes the input structure |
| Motor brick | `ChatDatabricks` | Connects to the Databricks serving endpoint |
| Finishing piece | `StrOutputParser` / `JsonOutputParser` | Converts raw output into usable format |
| Snapping bricks together | Pipe operator `\|` | Chains components left-to-right |
| Pressing "Go" | `.invoke()` / `.stream()` | Runs the assembled chain |

## Exam Alert

| # | Trap | Correct Answer |
|---|---|---|
| 1 | "LangChain requires the OpenAI SDK to call Databricks models" | **WRONG** -- Databricks provides `ChatDatabricks` in the `databricks-langchain` package. No OpenAI SDK needed. |
| 2 | "Use `ChatOpenAI` for Databricks endpoints" | **WRONG** -- Use `ChatDatabricks(endpoint='...')`. `ChatOpenAI` is for the OpenAI API, not Databricks. |
| 3 | "LCEL uses method chaining like `.then().then()`" | **WRONG** -- LCEL uses the pipe operator: `prompt \| llm \| parser`. This is inspired by Unix pipes, not JavaScript promises. |
| 4 | "You must manage API keys manually in LangChain on Databricks" | **WRONG** -- `ChatDatabricks` uses notebook authentication automatically. It reads `DATABRICKS_HOST` and `DATABRICKS_TOKEN` from the environment. |
| 5 | "LangChain chains can only be invoked synchronously" | **WRONG** -- LCEL chains support `.invoke()`, `.stream()`, `.batch()`, and `.ainvoke()` (async) out of the box. |

> **Exam tip:** When you see a question about "building a chain on Databricks," the answer almost always involves `ChatDatabricks` + LCEL pipe syntax. If the answer options include `ChatOpenAI`, that is the trap.

## Prerequisites & UI Navigation

### Prerequisites
- A running cluster attached to this notebook
- Access to Foundation Model APIs (available on most Databricks workspaces)
- The `databricks-langchain` package (installed in the next cell)

### How to Verify Your Serving Endpoint

Before we begin, confirm that the model endpoint is available:

1. **UI -->** Left navigation bar --> click **Serving** (the rocket icon)
2. Look for `databricks-meta-llama-3-3-70b-instruct` in the list of endpoints
3. Status should show **Ready** (green checkmark)
4. If you do not see it, you are likely on a workspace without Foundation Model APIs -- contact your workspace admin

### How to Browse Models in the Catalog

1. **UI -->** Left navigation bar --> click **Catalog**
2. Navigate to **System** --> **ai** --> **built_in** to see built-in Foundation Models
3. These are the same models available through Foundation Model APIs

---

## Setup

In [ ]:
# ------------------------------------------------------------------
# Setup: Install LangChain and the Databricks integration package
# ------------------------------------------------------------------
# databricks-langchain: provides ChatDatabricks, DatabricksEmbeddings
# langchain:            core LCEL framework, prompt templates, parsers
# langchain-community:  additional integrations and utilities
# ------------------------------------------------------------------

%pip install databricks-langchain langchain langchain-community --quiet

# Restart Python to pick up the new packages
dbutils.library.restartPython()

**Expected output:** Installation logs followed by `Python interpreter will be restarted.`

This is normal -- proceed to the next cell.

In [ ]:
# ------------------------------------------------------------------
# Setup: Import LangChain components and initialize ChatDatabricks
# ------------------------------------------------------------------

from databricks_langchain import ChatDatabricks
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
import json

# Initialize the LLM -- ChatDatabricks wraps a Databricks serving endpoint
# No API key needed: it uses notebook auth (DATABRICKS_HOST + DATABRICKS_TOKEN)
MODEL_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

llm = ChatDatabricks(
    endpoint=MODEL_ENDPOINT,
    temperature=0.0,
    max_tokens=512
)

print(f"ChatDatabricks ready. Endpoint: {MODEL_ENDPOINT}")
print(f"Type: {type(llm).__name__}")

**Expected output:**
```
ChatDatabricks ready. Endpoint: databricks-meta-llama-3-3-70b-instruct
Type: ChatDatabricks
```

**What just happened:** We imported the core LangChain components and created a `ChatDatabricks` instance. Notice we did not provide any API key or URL -- `ChatDatabricks` automatically picks up the notebook's authentication credentials. This is the Databricks-native approach to using LangChain.

---

## Step 1: ChatDatabricks Basics

**What it is:** `ChatDatabricks` is the LangChain `ChatModel` wrapper for Databricks model serving endpoints. It translates LangChain's standard interface (`.invoke()`, `.stream()`, `.batch()`) into Databricks API calls.

**Key point for the exam:** `ChatDatabricks` is imported from `databricks_langchain`, NOT from `langchain_community`. The `databricks-langchain` package is the official, maintained integration.

In [ ]:
# ------------------------------------------------------------------
# Step 1: Basic ChatDatabricks invocation
# ------------------------------------------------------------------
# .invoke() sends a single request and returns an AIMessage object.
# You can pass a plain string or a list of message objects.
# ------------------------------------------------------------------

response = llm.invoke("What is Unity Catalog in one sentence?")

print("=== ChatDatabricks Response ===")
print(f"Type:    {type(response).__name__}")
print(f"Content: {response.content}")
print()
print("=== Full AIMessage Object ===")
print(response)

**Expected output:**
```
=== ChatDatabricks Response ===
Type:    AIMessage
Content: Unity Catalog is a unified governance solution for all data and AI
         assets across multiple Databricks workspaces.

=== Full AIMessage Object ===
content='Unity Catalog is a unified governance solution...'
additional_kwargs={...} response_metadata={...}
```

**What just happened:** When you call `llm.invoke()` with a string, `ChatDatabricks` sends it to the Databricks serving endpoint and returns a LangChain `AIMessage` object. The actual text is in `.content`. This is important -- the raw return type is `AIMessage`, not a plain string. That is why we need output parsers (Step 4) to extract the text for downstream use.

---

## Step 2: Prompt Templates

**What it is:** `ChatPromptTemplate` lets you define reusable prompt structures with variables. Instead of f-strings with hardcoded text, you create a template once and fill it with different values each time.

**Why it matters:**
- Separates prompt logic from application logic
- Makes prompts testable, versionable, and reusable
- Works seamlessly with LCEL chains (the template becomes the first component)
- Supports system/human/AI message roles

In [ ]:
# ------------------------------------------------------------------
# Step 2: Create and inspect a ChatPromptTemplate
# ------------------------------------------------------------------

# Define a template with system + human messages and a variable {topic}
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Databricks expert. Give concise, accurate answers."),
    ("human", "Explain {topic} in exactly 2 sentences.")
])

# Inspect: what does the template look like before filling in variables?
print("=== Template Structure ===")
print(prompt)
print()

# Format the messages with a specific topic
formatted = prompt.format_messages(topic="Unity Catalog")
print("=== Formatted Messages ===")
for msg in formatted:
    print(f"  [{msg.type}] {msg.content}")
print()

# Invoke the LLM directly with the formatted messages
response = llm.invoke(formatted)
print("=== LLM Response ===")
print(response.content)

**Expected output:**
```
=== Template Structure ===
input_variables=['topic']
messages=[SystemMessagePromptTemplate(...), HumanMessagePromptTemplate(...)]

=== Formatted Messages ===
  [system] You are a Databricks expert. Give concise, accurate answers.
  [human] Explain Unity Catalog in exactly 2 sentences.

=== LLM Response ===
Unity Catalog is Databricks' unified governance solution that provides
centralized access control, auditing, and data lineage across all data
and AI assets. It enables organizations to manage permissions and discover
data across multiple workspaces from a single place.
```

**What just happened:** We created a `ChatPromptTemplate` with a system message and a human message containing a `{topic}` variable. The `.format_messages()` method shows exactly what gets sent to the LLM. This is useful for debugging -- you can see the exact prompt before it goes to the model.

But notice we had to manually call `.format_messages()` and then `llm.invoke()` -- that is two separate steps. LCEL pipe syntax (Step 3) eliminates this boilerplate.

---

## Step 3: LCEL Pipe Syntax (THE EXAM PATTERN)

**This is the most important pattern in this lab.** The exam tests LCEL pipe syntax extensively.

**What it is:** LCEL uses the pipe operator `|` to chain components together. Output from the left side becomes input to the right side, just like Unix pipes.

```python
chain = prompt | llm | parser
```

This reads: "Format the prompt, pass it to the LLM, parse the output."

**Why it matters:**
- One line of code replaces multiple manual steps
- Each component is independent and reusable
- The chain supports `.invoke()`, `.stream()`, `.batch()`, and `.ainvoke()` automatically
- This is THE pattern the exam expects you to recognize

In [ ]:
# ------------------------------------------------------------------
# Step 3: Build a chain with LCEL pipe syntax
# ------------------------------------------------------------------
# chain = prompt | llm | parser
#
# How data flows:
#   1. prompt.invoke({"topic": "..."}) --> ChatPromptValue (formatted messages)
#   2. llm.invoke(ChatPromptValue)     --> AIMessage (raw LLM response)
#   3. parser.invoke(AIMessage)        --> str (plain text)
# ------------------------------------------------------------------

# Reuse the prompt template from Step 2
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Databricks expert. Give concise, accurate answers."),
    ("human", "Explain {topic} in exactly 2 sentences.")
])

# Build the chain: prompt -> LLM -> string parser
chain = prompt | llm | StrOutputParser()

# Invoke the chain with a topic
result = chain.invoke({"topic": "Delta Lake"})

print("=== LCEL Chain Result ===")
print(f"Type:   {type(result).__name__}")
print(f"Result: {result}")
print()

# The chain is reusable -- invoke it with different inputs
topics = ["MLflow", "Feature Serving", "Vector Search"]
print("=== Chain Reuse with Multiple Topics ===")
for topic in topics:
    answer = chain.invoke({"topic": topic})
    print(f"  {topic}: {answer}")
    print()

**Expected output:**
```
=== LCEL Chain Result ===
Type:   str
Result: Delta Lake is an open-source storage layer that brings ACID
        transactions, scalable metadata handling, and schema enforcement
        to data lakes. It enables reliable data pipelines by supporting
        time travel, schema evolution, and unified batch and streaming
        processing.

=== Chain Reuse with Multiple Topics ===
  MLflow: MLflow is an open-source platform for managing the end-to-end
  machine learning lifecycle, including experiment tracking, model
  packaging, and deployment...

  Feature Serving: Feature Serving in Databricks provides real-time
  access to precomputed features for ML models...

  Vector Search: Vector Search is a Databricks service that enables
  similarity-based retrieval of documents and data using vector
  embeddings...
```

**What just happened:** The chain `prompt | llm | StrOutputParser()` does three things in sequence:

1. `prompt` takes `{"topic": "Delta Lake"}` and formats it into system + human messages
2. `llm` sends those messages to the Databricks endpoint and returns an `AIMessage`
3. `StrOutputParser()` extracts the `.content` string from the `AIMessage`

The result is a plain Python `str` -- ready to use in downstream code. Notice how we reused the same chain for multiple topics. This composability is the core value proposition of LCEL.

> **Key insight for the exam:** The pipe operator `|` is NOT method chaining (`.then().then()`). It is operator overloading -- LangChain components implement `__or__` to enable the pipe syntax. The exam may present `.then()` syntax as a trap answer.

---

## Step 4: Output Parsers

**What they are:** Output parsers convert the raw `AIMessage` from the LLM into a usable Python object. LangChain provides several parsers:

| Parser | Returns | Use When |
|---|---|---|
| `StrOutputParser` | `str` | You need plain text (most common) |
| `JsonOutputParser` | `dict` | You need structured data for programmatic processing |
| `PydanticOutputParser` | Pydantic model | You need validated, typed objects |

**Exam distinction:** Without a parser, the chain returns an `AIMessage` object. With `StrOutputParser()`, it returns a `str`. With `JsonOutputParser()`, it returns a `dict`. The exam tests whether you know which parser to use.

In [ ]:
# ------------------------------------------------------------------
# Step 4a: StrOutputParser -- returns a plain string
# ------------------------------------------------------------------

str_chain = prompt | llm | StrOutputParser()
str_result = str_chain.invoke({"topic": "Model Serving"})

print("=== StrOutputParser ===")
print(f"Type:   {type(str_result).__name__}")
print(f"Result: {str_result}")
print()

# ------------------------------------------------------------------
# Step 4b: Without a parser -- returns AIMessage
# ------------------------------------------------------------------

no_parser_chain = prompt | llm
raw_result = no_parser_chain.invoke({"topic": "Model Serving"})

print("=== Without Parser (raw AIMessage) ===")
print(f"Type:   {type(raw_result).__name__}")
print(f"Result: {raw_result}")
print()

# ------------------------------------------------------------------
# Why this matters: try using the raw result as a string
# ------------------------------------------------------------------
print("=== Practical Difference ===")
print(f"str_result[:50]:     '{str_result[:50]}...'")
try:
    print(f"raw_result[:50]:     '{raw_result[:50]}...'")
except TypeError as e:
    print(f"raw_result[:50]:     ERROR -- {e}")
    print("  --> AIMessage is not subscriptable. You need a parser!")

**Expected output:**
```
=== StrOutputParser ===
Type:   str
Result: Model Serving in Databricks provides a unified interface for
        deploying and serving ML and AI models as REST API endpoints...

=== Without Parser (raw AIMessage) ===
Type:   AIMessage
Result: content='Model Serving in Databricks provides...'
        additional_kwargs={...} response_metadata={...}

=== Practical Difference ===
str_result[:50]:     'Model Serving in Databricks provides a unified int...'
raw_result[:50]:     ERROR -- 'AIMessage' object is not subscriptable
  --> AIMessage is not subscriptable. You need a parser!
```

**What just happened:** Without a parser, the chain returns an `AIMessage` object that you cannot use directly as a string. The `StrOutputParser()` extracts the `.content` field and gives you a clean `str`. This is why almost every LCEL chain ends with an output parser.

In [ ]:
# ------------------------------------------------------------------
# Step 4c: JsonOutputParser -- returns a Python dict
# ------------------------------------------------------------------

json_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a structured data assistant. "
     "Always respond with valid JSON and nothing else. "
     "Do not wrap the JSON in markdown code fences."),
    ("human",
     "Extract the following from this Databricks concept description. "
     "Return a JSON object with keys: name, category, purpose, key_feature.\n\n"
     "Description: {description}")
])

json_chain = json_prompt | llm | JsonOutputParser()

result = json_chain.invoke({
    "description": (
        "Unity Catalog is a unified governance solution for all data and AI "
        "assets in Databricks. It provides centralized access control, auditing, "
        "lineage tracking, and data discovery across multiple workspaces."
    )
})

print("=== JsonOutputParser Result ===")
print(f"Type:   {type(result).__name__}")
print(f"Result: {json.dumps(result, indent=2)}")
print()

# Access individual fields -- this is why structured output matters
print("=== Accessing Fields Programmatically ===")
print(f"  Name:        {result.get('name', 'N/A')}")
print(f"  Category:    {result.get('category', 'N/A')}")
print(f"  Purpose:     {result.get('purpose', 'N/A')}")
print(f"  Key Feature: {result.get('key_feature', 'N/A')}")

**Expected output:**
```
=== JsonOutputParser Result ===
Type:   dict
Result: {
  "name": "Unity Catalog",
  "category": "Governance",
  "purpose": "Unified governance for data and AI assets",
  "key_feature": "Centralized access control across multiple workspaces"
}

=== Accessing Fields Programmatically ===
  Name:        Unity Catalog
  Category:    Governance
  Purpose:     Unified governance for data and AI assets
  Key Feature: Centralized access control across multiple workspaces
```

**What just happened:** `JsonOutputParser()` parses the LLM's raw text response into a Python `dict`. This is critical for production pipelines where downstream code needs to access individual fields. Compare this to `StrOutputParser()`, which would return the JSON as a string that you would have to parse manually with `json.loads()`.

> **When to use which parser:**
> - `StrOutputParser` -- when the consumer is a human (display text) or another LLM (pass text to next chain)
> - `JsonOutputParser` -- when the consumer is Python code that needs to access specific fields

---

## Step 5: Retriever Integration (RAG Chain Preview)

**What it is:** In a RAG (Retrieval-Augmented Generation) pipeline, a retriever fetches relevant documents from a vector store, and those documents become context for the LLM. LangChain makes this easy with `RunnablePassthrough`, which passes the original query through the chain alongside the retrieved documents.

**The full RAG chain pattern:**
```python
chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm | parser
```

This is a preview of the pattern you will build fully in Module 04 (Data Preparation). Here, we simulate a retriever with a simple function to focus on the chain mechanics.

**Key concept -- `RunnablePassthrough`:** When you need to pass the original input (e.g., the user's question) through to a later stage in the chain, use `RunnablePassthrough()`. It acts as an identity function -- it receives input and passes it through unchanged.

In [ ]:
# ------------------------------------------------------------------
# Step 5: Build a RAG-style chain with a simulated retriever
# ------------------------------------------------------------------

# Simulate a retriever -- in production this would be a Vector Search index
# We use a simple function that returns Document objects
def mock_retriever(query: str):
    """Simulates retrieving relevant documents from a vector store."""
    # In production: DatabricksVectorSearch or similar retriever
    knowledge_base = {
        "unity catalog": [
            Document(page_content="Unity Catalog provides centralized governance for data and AI assets across Databricks workspaces. It supports fine-grained access control at the table, column, and row level."),
            Document(page_content="Unity Catalog tracks data lineage automatically, showing how data flows from source tables through transformations to final outputs.")
        ],
        "model serving": [
            Document(page_content="Databricks Model Serving deploys ML and AI models as REST API endpoints with automatic scaling. It supports real-time and batch inference."),
            Document(page_content="Model Serving integrates with MLflow for model versioning and supports A/B testing through traffic routing between model versions.")
        ],
        "default": [
            Document(page_content="Databricks is a unified analytics platform built on Apache Spark, providing tools for data engineering, data science, and machine learning."),
            Document(page_content="Databricks Lakehouse combines the best of data warehouses and data lakes into a single platform.")
        ]
    }

    # Simple keyword matching (a real retriever uses vector similarity)
    for key in knowledge_base:
        if key in query.lower():
            return knowledge_base[key]
    return knowledge_base["default"]


# Define a RAG prompt template
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful Databricks assistant. Answer the question using ONLY "
     "the provided context. If the context does not contain the answer, say "
     "'I don't have enough information to answer that.'"),
    ("human",
     "Context:\n{context}\n\n"
     "Question: {question}\n\n"
     "Answer:")
])

# Helper to format retrieved documents into a single context string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Build the RAG chain
# RunnablePassthrough() passes the original input through as "question"
# The retriever function provides the "context"
rag_chain = (
    {
        "context": lambda x: format_docs(mock_retriever(x)),
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test the chain
question = "What governance features does Unity Catalog provide?"
answer = rag_chain.invoke(question)

print("=== RAG Chain Result ===")
print(f"Question: {question}")
print(f"Answer:   {answer}")
print()

# Test with a different topic
question2 = "How does model serving handle scaling?"
answer2 = rag_chain.invoke(question2)
print(f"Question: {question2}")
print(f"Answer:   {answer2}")

**Expected output:**
```
=== RAG Chain Result ===
Question: What governance features does Unity Catalog provide?
Answer:   Based on the provided context, Unity Catalog provides centralized
          governance for data and AI assets across Databricks workspaces,
          including fine-grained access control at the table, column, and
          row level, as well as automatic data lineage tracking.

Question: How does model serving handle scaling?
Answer:   According to the context, Databricks Model Serving deploys models
          as REST API endpoints with automatic scaling, supporting both
          real-time and batch inference.
```

**What just happened:** We built a complete RAG chain:

1. The input string goes to two places simultaneously:
   - `lambda x: format_docs(mock_retriever(x))` -- retrieves and formats relevant documents
   - `RunnablePassthrough()` -- passes the original question through unchanged
2. Both values feed into the prompt template as `{context}` and `{question}`
3. The formatted prompt goes to the LLM
4. `StrOutputParser()` extracts the text response

**The pattern to remember for the exam:**
```python
chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm | parser
```

In Module 04, you will replace the mock retriever with a real `DatabricksVectorSearch` retriever. The chain structure stays the same.

---

## Step 6: Streaming

**What it is:** Instead of waiting for the entire response to generate, streaming returns tokens one at a time as they are produced. This is critical for chatbot UX -- users see the response appear word-by-word rather than staring at a blank screen.

**How it works:** LCEL chains support `.stream()` out of the box. When you call `chain.stream(input)`, it returns a generator that yields chunks of the response as they become available.

**Why it matters on the exam:** The exam tests whether you know that LCEL chains support streaming natively. You do not need to implement custom streaming logic -- just call `.stream()` instead of `.invoke()`.

In [ ]:
# ------------------------------------------------------------------
# Step 6: Streaming -- token-by-token output
# ------------------------------------------------------------------

stream_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful Databricks expert."),
    ("human", "List 3 key benefits of {topic}. Keep each benefit to one sentence.")
])

stream_chain = stream_prompt | llm | StrOutputParser()

print("=== Streaming Output ===")
print("(tokens appear one at a time as the model generates them)")
print()

# .stream() returns a generator that yields chunks
for chunk in stream_chain.stream({"topic": "Delta Lake"}):
    print(chunk, end="", flush=True)

print()  # newline after streaming completes
print()

# --- Compare: .invoke() vs .stream() ---
print("=== Comparison ===")
print(".invoke() -- waits for full response, returns complete string")
print(".stream() -- yields chunks as they generate, shows output in real-time")
print()
print("Both produce the same final text. The difference is UX:")
print("  - Use .invoke() for batch processing and pipelines")
print("  - Use .stream() for interactive chat applications")

**Expected output:**
```
=== Streaming Output ===
(tokens appear one at a time as the model generates them)

1. **ACID Transactions**: Delta Lake provides ACID transaction support,
ensuring data reliability and consistency even with concurrent reads
and writes.
2. **Time Travel**: Delta Lake enables time travel, allowing users to
query and restore previous versions of data for auditing and debugging.
3. **Schema Enforcement**: Delta Lake enforces schema on write,
preventing bad data from corrupting your tables and ensuring data quality.

=== Comparison ===
.invoke() -- waits for full response, returns complete string
.stream() -- yields chunks as they generate, shows output in real-time

Both produce the same final text. The difference is UX:
  - Use .invoke() for batch processing and pipelines
  - Use .stream() for interactive chat applications
```

**What just happened:** When you call `.stream()`, the chain returns a generator. Each iteration yields a small chunk of text (often a few tokens). In a notebook, you see the text appear progressively. In a web application, you would send each chunk to the client via Server-Sent Events (SSE) or WebSockets.

> **Key insight:** You do NOT need to write custom streaming code. LCEL chains support `.stream()` natively because every component in the chain (prompt, LLM, parser) implements the streaming interface. This is a common exam point.

---

## Step 7: RunnableSequence — What the Pipe Operator Actually Creates

When you write `prompt | llm | parser`, LangChain creates a **`RunnableSequence`** object behind the scenes. This is the exam-tested name for "a series of steps executed in order, where each step's output becomes the next step's input."

### Exam Question Pattern

> *A developer wants to optimize latency in their RAG chain that uses multiple tools. What LangChain component should they use to manage sequential execution?*
>
> - (A) ToolExecutor
> - **(B) RunnableSequence** ← Correct
> - (C) PromptTemplate
> - (D) AsyncRetriever

### Why Each Distractor Is Wrong

| Option | What It Actually Is | Why It's Wrong |
|---|---|---|
| **ToolExecutor** | Not a standard LangChain component — agents handle tool execution internally | Does not manage sequential chain composition |
| **PromptTemplate** | A single component that formats prompts | It's one step *inside* the sequence, not the orchestrator |
| **AsyncRetriever** | Not a standard LangChain class | Retrieval is one step, not the sequencing mechanism |
| **RunnableSequence** | The LCEL orchestrator that chains steps in order | **Correct** — it defines the execution order and passes data between steps |

Let's prove this by inspecting what the pipe operator actually builds.

In [ ]:
# ------------------------------------------------------------------
# Step 7: Inspect the RunnableSequence created by the pipe operator
# ------------------------------------------------------------------

from langchain_core.runnables import RunnableSequence

# Build a chain with the pipe operator
chain = prompt | llm | StrOutputParser()

# What type is it?
print("=== What does the pipe operator create? ===")
print(f"Type: {type(chain).__name__}")
print(f"Is RunnableSequence? {isinstance(chain, RunnableSequence)}")
print()

# Inspect the steps inside the sequence
print("=== Steps in the RunnableSequence ===")
for i, step in enumerate(chain.steps if hasattr(chain, 'steps') else [chain.first, chain.middle[0] if chain.middle else None, chain.last]):
    if step is not None:
        print(f"  Step {i+1}: {type(step).__name__}")
print()

# You can also build a RunnableSequence explicitly (same result)
explicit_chain = RunnableSequence(
    first=prompt,
    middle=[llm],
    last=StrOutputParser()
)
print("=== Explicit RunnableSequence ===")
print(f"Type: {type(explicit_chain).__name__}")
print(f"Same result? Let's test...")
print()

# Both chains produce the same output
result_pipe = chain.invoke({"topic": "LCEL"})
result_explicit = explicit_chain.invoke({"topic": "LCEL"})

print(f"Pipe operator chain:     {result_pipe[:80]}...")
print(f"Explicit RunnableSeq:    {result_explicit[:80]}...")
print()
print("Both approaches create the same RunnableSequence.")
print("The pipe operator is just syntactic sugar.")


**Expected output:**
```
=== What does the pipe operator create? ===
Type: RunnableSequence
Is RunnableSequence? True

=== Steps in the RunnableSequence ===
  Step 1: ChatPromptTemplate
  Step 2: ChatDatabricks
  Step 3: StrOutputParser

=== Explicit RunnableSequence ===
Type: RunnableSequence
Same result? Let's test...

Pipe operator chain:     LCEL (LangChain Expression Language) is a declarative syntax...
Explicit RunnableSeq:    LCEL is a powerful framework for composing chains...

Both approaches create the same RunnableSequence.
The pipe operator is just syntactic sugar.
```

**What just happened:** You proved that `prompt | llm | parser` creates a `RunnableSequence` — the same class you can construct explicitly. The pipe operator is syntactic sugar that makes chains readable.

### Why RunnableSequence Optimizes Latency

RunnableSequence is not just about ordering — it provides:

- **Streaming propagation** — if one step supports streaming, the entire sequence streams (no buffering between steps)
- **Batch optimization** — `.batch()` runs multiple inputs through the sequence efficiently
- **Async support** — `.ainvoke()` and `.astream()` enable non-blocking execution
- **Error propagation** — failures in any step surface immediately with clear tracebacks

This is why the exam answer is RunnableSequence: it's the **orchestration layer** that manages execution order, data flow, and performance optimizations across all steps in the chain.

> **Exam tip:** `prompt | llm | parser` = `RunnableSequence(prompt, llm, parser)`. When the exam asks about "sequential execution" or "managing step order" in LangChain, the answer is **RunnableSequence**.

---

## Step 8: Callback Handlers — Observing What Happens Inside a Chain

When a chain runs, a lot happens invisibly: the prompt gets formatted, the LLM receives tokens, the parser processes output. **Callback handlers** let you hook into these events — for logging, debugging, monitoring, or custom telemetry.

### What Are Callbacks?

A callback handler is a class that receives notifications at each stage of chain execution:

| Event | When It Fires | Use Case |
|---|---|---|
| `on_chain_start` | Chain begins executing | Log the input, start a timer |
| `on_chain_end` | Chain finishes | Log the output, record latency |
| `on_llm_start` | LLM call begins | Log the prompt sent to the model |
| `on_llm_end` | LLM call finishes | Log token usage, response time |
| `on_llm_new_token` | Each token arrives (streaming) | Update a progress bar, stream to UI |
| `on_chain_error` | An error occurs | Alert, retry, log for debugging |
| `on_retriever_start` / `on_retriever_end` | Retriever runs | Log which documents were retrieved |

### Why This Matters

- **Debugging**: See exactly what prompt was sent and what came back
- **Monitoring**: Track latency, token usage, and error rates in production
- **Cost tracking**: Count tokens per call for billing visibility
- **MLflow integration**: Databricks uses callbacks internally for MLflow Tracing

> **Analogy:** Callbacks are like security cameras in a factory. The assembly line (chain) runs the same whether cameras are on or off, but with cameras you can see exactly what happened at each station, how long each step took, and where things went wrong.

In [ ]:
# ------------------------------------------------------------------
# Step 8a: Build a custom CallbackHandler
# ------------------------------------------------------------------

from langchain_core.callbacks import BaseCallbackHandler
from datetime import datetime


class LabCallbackHandler(BaseCallbackHandler):
    """
    Custom callback handler that logs chain and LLM events.
    In production, you'd send these to a monitoring system.
    Here, we print them to understand what happens inside a chain.
    """

    def __init__(self):
        self.chain_start_time = None
        self.llm_start_time = None
        self.token_count = 0

    def on_chain_start(self, serialized, inputs, **kwargs):
        self.chain_start_time = datetime.now()
        chain_name = serialized.get("name", "unknown")
        print(f"  [CHAIN START]  {chain_name}")
        # Only show input preview for the top-level chain
        if isinstance(inputs, dict):
            for k, v in inputs.items():
                print(f"                 input.{k} = {str(v)[:60]}")

    def on_chain_end(self, outputs, **kwargs):
        elapsed = (datetime.now() - self.chain_start_time).total_seconds()
        print(f"  [CHAIN END]    completed in {elapsed:.2f}s")

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.llm_start_time = datetime.now()
        self.token_count = 0
        model = serialized.get("name", serialized.get("id", ["unknown"])[-1])
        print(f"  [LLM START]    model={model}")

    def on_llm_end(self, response, **kwargs):
        elapsed = (datetime.now() - self.llm_start_time).total_seconds()
        # Try to get token usage from response metadata
        usage = ""
        if response.generations and response.generations[0]:
            gen = response.generations[0][0]
            if hasattr(gen, "generation_info") and gen.generation_info:
                token_info = gen.generation_info.get("token_usage", {})
                if token_info:
                    usage = f" | tokens: {token_info}"
        print(f"  [LLM END]      completed in {elapsed:.2f}s{usage}")

    def on_chain_error(self, error, **kwargs):
        print(f"  [CHAIN ERROR]  {type(error).__name__}: {error}")


print("LabCallbackHandler defined.")
print("Events: on_chain_start, on_chain_end, on_llm_start, on_llm_end, on_chain_error")


In [ ]:
# ------------------------------------------------------------------
# Step 8b: Run a chain WITH the callback handler
# ------------------------------------------------------------------

# Create the handler
handler = LabCallbackHandler()

# Build a simple chain
observed_chain = prompt | llm | StrOutputParser()

# Pass the callback via the `config` parameter
# This is the standard way to attach callbacks in LCEL
print("=" * 60)
print("Running chain WITH callback handler:")
print("=" * 60)

result = observed_chain.invoke(
    {"topic": "Vector Search"},
    config={"callbacks": [handler]}
)

print()
print("=== Final Output ===")
print(result[:150] + "...")
print()

# ---- Now run WITHOUT a callback for comparison ----
print("=" * 60)
print("Running chain WITHOUT callback handler:")
print("=" * 60)

result2 = observed_chain.invoke({"topic": "Vector Search"})
print(f"Result: {result2[:150]}...")
print()
print("Notice: same result, but no visibility into what happened inside.")
print("Callbacks are observability — they don't change the chain's behavior.")


**Expected output:**
```
============================================================
Running chain WITH callback handler:
============================================================
  [CHAIN START]  RunnableSequence
                 input.topic = Vector Search
  [CHAIN START]  ChatPromptTemplate
  [CHAIN END]    completed in 0.00s
  [LLM START]    model=ChatDatabricks
  [LLM END]      completed in 1.23s
  [CHAIN START]  StrOutputParser
  [CHAIN END]    completed in 0.00s
  [CHAIN END]    completed in 1.24s

=== Final Output ===
Vector Search is a Databricks service that enables similarity-based...

============================================================
Running chain WITHOUT callback handler:
============================================================
Result: Vector Search is a Databricks service that enables...

Notice: same result, but no visibility into what happened inside.
Callbacks are observability — they don't change the chain's behavior.
```

**What just happened:** The callback handler received events for every step in the chain: the RunnableSequence started, then each sub-step (prompt, LLM, parser) fired its own start/end events. You can see exactly how long the LLM call took vs the total chain time. Without the callback, the chain runs identically but you get zero visibility.

### Attaching Callbacks — Two Methods

| Method | Syntax | Scope |
|---|---|---|
| **Per-invocation** (recommended) | `chain.invoke(input, config={"callbacks": [handler]})` | Only this call |
| **On the chain** | `chain.with_config(callbacks=[handler]).invoke(input)` | Every call on this chain |

The `config={"callbacks": [...]}` approach is preferred because it keeps the chain definition clean and lets you toggle observability on/off per call.

In [ ]:
# ------------------------------------------------------------------
# Step 8c: Practical use case — a cost-tracking callback
# ------------------------------------------------------------------

class CostTracker(BaseCallbackHandler):
    """
    Tracks cumulative token usage and estimated cost across
    multiple chain invocations. In production, you'd send
    these metrics to a dashboard or alerting system.
    """

    def __init__(self):
        self.total_calls = 0
        self.total_input_tokens = 0
        self.total_output_tokens = 0

    def on_llm_end(self, response, **kwargs):
        self.total_calls += 1
        # Extract token usage from response if available
        if response.generations and response.generations[0]:
            gen = response.generations[0][0]
            info = getattr(gen, "generation_info", {}) or {}
            usage = info.get("token_usage", {})
            self.total_input_tokens += usage.get("prompt_tokens", 0)
            self.total_output_tokens += usage.get("completion_tokens", 0)

    def report(self):
        total_tokens = self.total_input_tokens + self.total_output_tokens
        print("=== Cost Tracking Report ===")
        print(f"  Total LLM calls:    {self.total_calls}")
        print(f"  Input tokens:       {self.total_input_tokens}")
        print(f"  Output tokens:      {self.total_output_tokens}")
        print(f"  Total tokens:       {total_tokens}")
        if total_tokens > 0:
            # Approximate cost for pay-per-token Foundation Model APIs
            est_cost = total_tokens * 0.000002  # rough estimate
            print(f"  Est. cost:          ${est_cost:.6f}")
        else:
            print(f"  (Token counts not available in response metadata)")


# Use the cost tracker across multiple calls
tracker = CostTracker()

topics = ["Delta Lake", "MLflow", "Unity Catalog", "Feature Serving"]
print("Running 4 chain invocations with cost tracking...\n")

for topic in topics:
    result = observed_chain.invoke(
        {"topic": topic},
        config={"callbacks": [tracker]}
    )
    print(f"  {topic}: {result[:60]}...")

print()
tracker.report()
print()
print("KEY INSIGHT: Callbacks let you track cost, latency, and usage")
print("across many invocations — essential for production monitoring.")


**Expected output:**
```
Running 4 chain invocations with cost tracking...

  Delta Lake: Delta Lake is an open-source storage layer that brings ACID...
  MLflow: MLflow is an open-source platform for managing the end-to-end...
  Unity Catalog: Unity Catalog is Databricks' unified governance solution...
  Feature Serving: Feature Serving in Databricks provides real-time access...

=== Cost Tracking Report ===
  Total LLM calls:    4
  Input tokens:       ~140
  Output tokens:      ~280
  Total tokens:       ~420
  Est. cost:          $0.000840
```

(Exact numbers depend on model response length and whether token counts are exposed in metadata.)

**What just happened:** The `CostTracker` callback accumulated token usage across 4 separate chain invocations. In production, you'd send these metrics to a monitoring dashboard, set alerts for cost thresholds, or log them to an inference table (Lab 06-03).

### Built-in and Common Callback Handlers

LangChain provides several ready-made handlers:

| Handler | What It Does | Import |
|---|---|---|
| `StdOutCallbackHandler` | Prints all events to stdout (debugging) | `from langchain_core.callbacks import StdOutCallbackHandler` |
| `BaseCallbackHandler` | Base class for custom handlers (extend this) | `from langchain_core.callbacks import BaseCallbackHandler` |
| MLflow Tracing | Automatic tracing in Databricks (no code needed) | Enabled via `mlflow.langchain.autolog()` |

> **Databricks integration:** On Databricks, MLflow Tracing uses callbacks internally to capture chain execution traces. When you enable `mlflow.langchain.autolog()`, it automatically attaches a callback handler that logs every chain step to your MLflow experiment — no custom handler needed.

> **Exam tip:** If a question asks about "observing" or "monitoring" chain execution, the answer involves **callback handlers**. They are the mechanism for hooking into chain events without modifying the chain itself.

---

## Exam Tips & Traps

**Quick-scan reference -- review these before the exam:**

| # | Topic | Trap | Correct Answer |
|---|---|---|---|
| 1 | LLM wrapper | "Use `ChatOpenAI` to call Databricks endpoints" | **WRONG** -- Use `ChatDatabricks(endpoint='...')` from `databricks_langchain`. `ChatOpenAI` is for the OpenAI API. |
| 2 | LCEL syntax | "LCEL chains use method chaining: `prompt.then(llm).then(parser)`" | **WRONG** -- LCEL uses the pipe operator: `prompt \| llm \| parser`. The `\|` passes output left-to-right. |
| 3 | Output parsers | "LangChain chains always return strings" | **WRONG** -- Without a parser, chains return `AIMessage`. Use `StrOutputParser()` for strings or `JsonOutputParser()` for dicts. |
| 4 | Authentication | "You need to configure API keys for ChatDatabricks" | **WRONG** -- `ChatDatabricks` uses notebook auth automatically (`DATABRICKS_HOST` + `DATABRICKS_TOKEN`). |
| 5 | Streaming | "Streaming requires custom WebSocket implementation" | **WRONG** -- LCEL chains support `.stream()` natively. Just replace `.invoke()` with `.stream()`. |
| 6 | RAG chains | "The retriever replaces the prompt in a RAG chain" | **WRONG** -- The retriever provides *context* to the prompt. The chain is: `retriever \| prompt \| llm \| parser`. |
| 7 | RunnablePassthrough | "RunnablePassthrough transforms the input" | **WRONG** -- `RunnablePassthrough()` passes input through *unchanged*. It is the identity function for chains. |
| 8 | Package import | "`ChatDatabricks` is in `langchain_community`" | **WRONG** -- `ChatDatabricks` is in `databricks_langchain` (the official Databricks package). |

---

## Quick Knowledge Check

Test yourself before moving on. Answers are in the next cell.

1. What is the LCEL pipe syntax for a basic chain with a prompt, LLM, and string parser?
2. What Python type does `StrOutputParser()` return? What about `JsonOutputParser()`?
3. What does `RunnablePassthrough()` do in a RAG chain?
4. Where do you import `ChatDatabricks` from?
5. What is the difference between `.invoke()` and `.stream()`?

In [ ]:
# ------------------------------------------------------------------
# Knowledge Check Answers (run this cell to see answers)
# ------------------------------------------------------------------

answers = {
    "1": "chain = prompt | llm | StrOutputParser()",
    "2": "StrOutputParser -> str, JsonOutputParser -> dict",
    "3": "Passes the original input through unchanged (identity function)",
    "4": "from databricks_langchain import ChatDatabricks",
    "5": ".invoke() waits for full response; .stream() yields chunks in real-time"
}

for q, a in answers.items():
    print(f"  Q{q}: {a}")

**Expected output:**
```
  Q1: chain = prompt | llm | StrOutputParser()
  Q2: StrOutputParser -> str, JsonOutputParser -> dict
  Q3: Passes the original input through unchanged (identity function)
  Q4: from databricks_langchain import ChatDatabricks
  Q5: .invoke() waits for full response; .stream() yields chunks in real-time
```

---

## Key Takeaways

### Core Concepts
- **LCEL pipe syntax** (`|`) composes chains left-to-right: `prompt | llm | parser`. Output from each component becomes input to the next.
- **`ChatPromptTemplate`** creates reusable prompts with variables and message roles (system, human, AI).
- **`StrOutputParser`** converts `AIMessage` to `str`. **`JsonOutputParser`** converts to `dict`. Without a parser, you get raw `AIMessage` objects.
- **`RunnablePassthrough()`** is the identity function -- it passes input through unchanged. Essential for RAG chains where the query must reach the prompt template.
- **Streaming** (`.stream()`) is built into every LCEL chain. No custom implementation needed.

### Databricks-Specific
- **`ChatDatabricks`** is the official LangChain wrapper, imported from `databricks_langchain` (NOT `langchain_community`).
- **No API keys needed** -- `ChatDatabricks` uses notebook auth automatically via `DATABRICKS_HOST` and `DATABRICKS_TOKEN`.
- **Any serving endpoint** works -- Foundation Model APIs, fine-tuned models, external models behind AI Gateway.
- **Verify endpoints** at **UI -->** Left nav --> **Serving** to confirm models are ready.

### Exam Essentials
- **"Build a chain on Databricks"** --> `ChatDatabricks` + LCEL pipe syntax
- **"Parse LLM output as text"** --> `StrOutputParser()`
- **"Parse LLM output as JSON"** --> `JsonOutputParser()`
- **"Pass the original query through a RAG chain"** --> `RunnablePassthrough()`
- **"Enable real-time output"** --> `.stream()` (built into LCEL)
- **LCEL uses `|`, NOT `.then()`** -- this is a common trap
- **`ChatDatabricks`, NOT `ChatOpenAI`** -- for Databricks endpoints

---

## Next Lab

**Lab 03-02: Prompt Augmentation** -- you will learn how to enhance prompts with external context, build retrieval-augmented prompts, and implement dynamic prompt construction patterns. This builds directly on the LCEL chain patterns you learned here, adding sophisticated prompt engineering techniques for production applications.